# Lab 11B — Column-level Security, Row Filters y Tags en Unity Catalog

**Sesion 11 | Databricks Data Engineer Associate**  

**Runtime minimo:** DBR 13.3 LTS  
**Archivo fuente:** `transacciones_financieras.csv`

## Objetivos
- Aplicar tags a tablas y columnas para catalogacion y auditoria de PII
- Crear funciones de Column Masking para ocultar datos sensibles segun el rol del usuario
- Implementar Row Filters para segmentar el acceso por pais
- Auditar los tags con information_schema

## Setup previo
Subir `transacciones_financieras.csv` al Volume:  
`/Volumes/dbassociate/default/vol_landing/`

## Paso 0 — Verificacion del entorno

In [0]:
display(dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/"))

In [0]:
CATALOG       = "dbassociate"
SCHEMA_SILVER = "silver"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing/sesion11"
SOURCE_FILE   = f"{VOLUME_PATH}/transacciones_financieras.csv"
TABLE_PATH = "abfss://<container-name>@<storage-account-name>.dfs.core.windows.net/<path-to-data>"


print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA_SILVER}")
print(f"Archivo : {SOURCE_FILE}")

## Paso 1 — Cargar datos de transacciones financieras

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema_tx = StructType([
    StructField("transaccion_id",   StringType(), False),
    StructField("cuenta_origen",    StringType(), True),
    StructField("cuenta_destino",   StringType(), True),
    StructField("monto",            DoubleType(), True),
    StructField("tipo_transaccion", StringType(), True),
    StructField("fecha",            StringType(), True),
    StructField("hora",             StringType(), True),
    StructField("pais",             StringType(), True),
    StructField("estado",           StringType(), True),
])

df_tx = (
    spark.read
    .option("header", True)
    .schema(schema_tx)
    .csv(SOURCE_FILE)
)

df_tx.printSchema()
display(df_tx)

## Paso 2 — Crear tabla en la capa Silver

In [0]:
(
    df_tx.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.transacciones_financieras")
)

print(f"Tabla creada: {CATALOG}.{SCHEMA_SILVER}.transacciones_financieras")
spark.sql("SELECT COUNT(*) AS total FROM dbassociate.silver.transacciones_financieras").display()

## Paso 3 — Agregar TAGS y COMMENT a la tabla

Los tags son pares clave-valor que permiten categorizar objetos UC para busqueda,
auditoria de PII y cumplimiento normativo (GDPR, CCPA, SOX).

In [0]:
# Tags a nivel de tabla
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    SET TAGS (
        'dominio'        = 'finanzas',
        'clasificacion'  = 'confidencial',
        'pii'            = 'true',
        'propietario'    = 'equipo_finanzas'
    )
""")
print("Tags de tabla aplicados.")

In [0]:
# Comentario descriptivo a nivel de tabla
spark.sql("""
    COMMENT ON TABLE dbassociate.silver.transacciones_financieras
    IS 'Transacciones financieras procesadas en capa silver. Contiene numeros de cuenta — acceso restringido por politica de datos.'
""")
print("Comentario de tabla aplicado.")

In [0]:
# Tags a nivel de columna para columnas PII
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    ALTER COLUMN cuenta_origen
    SET TAGS ('pii' = 'true', 'tipo_pii' = 'numero_cuenta')
""")

spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    ALTER COLUMN cuenta_destino
    SET TAGS ('pii' = 'true', 'tipo_pii' = 'numero_cuenta')
""")

print("Tags de columnas PII aplicados.")

In [0]:
# Verificar los tags aplicados a la tabla
spark.sql("""
    SELECT 'TABLA' AS nivel, NULL AS columna, tag_name, tag_value
    FROM dbassociate.information_schema.table_tags
    WHERE catalog_name = 'dbassociate'
      AND schema_name = 'silver'
      AND table_name = 'transacciones_financieras'
    
    UNION ALL
    
    SELECT 'COLUMNA' AS nivel, column_name AS columna, tag_name, tag_value
    FROM dbassociate.information_schema.column_tags
    WHERE catalog_name = 'dbassociate'
      AND schema_name = 'silver'
      AND table_name = 'transacciones_financieras'
    ORDER BY nivel DESC, columna, tag_name
""").display()

## Paso 4 — Column Masking: ocultar numeros de cuenta segun el rol

La funcion de mascara recibe el valor original y retorna el valor visible.  
`IS_ACCOUNT_GROUP_MEMBER` verifica si el usuario pertenece a un grupo de la cuenta Databricks.

In [0]:
# Crear la funcion de column masking en el schema silver
spark.sql("""
    CREATE OR REPLACE FUNCTION dbassociate.silver.mask_cuenta(cuenta STRING)
    RETURNS STRING
    RETURN CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('data-engineer') THEN cuenta
        ELSE CONCAT('****-****-****-', RIGHT(cuenta, 4))
    END
""")
print("Funcion de mascara creada: dbassociate.silver.mask_cuenta")

In [0]:
# Probar la funcion antes de asociarla a la tabla
# Como data_engineer: vera el numero completo
# Como analista: veria ****-****-****-6789
spark.sql("""
    SELECT
        '4532-7891-2345-6789'                               AS numero_real,
        dbassociate.silver.mask_cuenta('4532-7891-2345-6789') AS numero_enmascarado
""").display()

In [0]:
# Asociar la mascara a la columna cuenta_origen
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    ALTER COLUMN cuenta_origen
    SET MASK dbassociate.silver.mask_cuenta
""")

# Asociar la misma mascara a cuenta_destino
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    ALTER COLUMN cuenta_destino
    SET MASK dbassociate.silver.mask_cuenta
""")

print("Column masks aplicadas a cuenta_origen y cuenta_destino.")

In [0]:
# El valor visible en el SELECT depende del grupo del usuario que ejecuta la query
# Un data_engineer ve el numero completo; un analista ve ****-****-****-XXXX
spark.sql("""
    SELECT transaccion_id, cuenta_origen, cuenta_destino, monto, pais
    FROM dbassociate.silver.transacciones_financieras
    LIMIT 5
""").display()

## Paso 5 — Row Filter: segmentar acceso por pais

El row filter aplica un predicado transparente a cada query sobre la tabla.  
El usuario no puede bypassearlo — es aplicado por el motor de UC antes de retornar resultados.

In [0]:
# La funcion retorna TRUE para las filas que el usuario puede ver
# data_engineers ven todas las filas
# Todos los demas solo ven filas con pais = 'Mexico'
spark.sql("""
    CREATE OR REPLACE FUNCTION dbassociate.silver.filter_pais_mx(pais STRING)
    RETURNS BOOLEAN
    RETURN
        IS_ACCOUNT_GROUP_MEMBER('GRP_TEST')
        OR pais = 'Mexico'
""")
print("Row filter creado: dbassociate.silver.filter_pais_mx")

In [0]:
# Asociar el row filter a la tabla
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    SET ROW FILTER dbassociate.silver.filter_pais_mx ON (pais)
""")
print("Row filter aplicado a la tabla.")

In [0]:
# Ver distribucion por pais
# Un data_engineer vera los tres paises
# Un analista sin el grupo 'data_engineers' solo vera filas de Mexico
spark.sql("""
    SELECT pais, COUNT(*) AS total_transacciones, ROUND(SUM(monto), 2) AS monto_total
    FROM dbassociate.silver.transacciones_financieras
    GROUP BY pais
    ORDER BY monto_total DESC
""").display()

## Paso 6 — Auditar PII con information_schema

Una vez que las columnas PII estan etiquetadas, se puede generar un inventario
completo de datos sensibles usando SQL sobre el information_schema.

In [0]:
# Inventario de columnas etiquetadas como PII en todo el catalog dbassociate
spark.sql("""
    SELECT
        c.table_schema     AS schema_name,
        c.table_name,
        c.column_name,
        c.data_type,
        ct.tag_name,
        ct.tag_value
    FROM dbassociate.information_schema.columns c
    JOIN dbassociate.information_schema.column_tags ct
        ON  ct.catalog_name = c.table_catalog
        AND ct.schema_name  = c.table_schema
        AND ct.table_name   = c.table_name
        AND ct.column_name  = c.column_name
    WHERE ct.tag_name    = 'pii'
      AND ct.tag_value   = 'true'
      AND c.table_catalog = 'dbassociate'
    ORDER BY c.table_schema, c.table_name, c.column_name
""").display()

In [0]:
# Inventario de tablas clasificadas como confidenciales
spark.sql("""
    SELECT
        tt.schema_name,
        tt.table_name,
        tt.tag_name,
        tt.tag_value
    FROM dbassociate.information_schema.table_tags tt
    WHERE tt.catalog_name  = 'dbassociate'
      AND tt.tag_name       = 'clasificacion'
    ORDER BY tt.schema_name, tt.table_name
""").display()

## Paso 7 — Quitar mask y row filter antes de limpiar

In [0]:
# Quitar column masks
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    ALTER COLUMN cuenta_origen DROP MASK
""")
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    ALTER COLUMN cuenta_destino DROP MASK
""")

# Quitar row filter
spark.sql("""
    ALTER TABLE dbassociate.silver.transacciones_financieras
    DROP ROW FILTER
""")

print("Column masks y row filter removidos.")

## Paso 8 — Limpieza

In [0]:
spark.sql("DROP TABLE    IF EXISTS dbassociate.silver.transacciones_financieras")
spark.sql("DROP FUNCTION IF EXISTS dbassociate.silver.mask_cuenta")
spark.sql("DROP FUNCTION IF EXISTS dbassociate.silver.filter_pais_mx")
print("Limpieza completada.")

## Puntos clave del examen

1. **Column Masking**: funcion SQL que transforma el valor de una columna segun el rol — el query del usuario no cambia
2. **Row Filter**: predicado SQL que limita las filas visibles — aplicado transparentemente por UC antes de retornar resultados
3. La diferencia: masking transforma **valores**, row filter limita **filas**
4. Ambas funciones se registran como UDFs en UC y se asocian a la tabla con `ALTER TABLE`
5. `IS_ACCOUNT_GROUP_MEMBER('grupo')` es la funcion UC para discriminar por rol
6. Los tags de columna son auditables via `information_schema.column_tags`